# Instant3D Gradio Colab — ROI候補検索対応版 v2

TotalSegmentatorをColab上で実行し、ROI候補を検索・複数選択して、STL / SegRef3D互換label PNG / 体積CSVなどを出力します。

**主な改善点**

- ROI名を手入力しなくても、taskに応じた候補から検索・複数選択できます。
- 添付の `resources/roi_catalog_*.txt` 相当のカタログをノートブック内に埋め込み済みです。
- STL、SegRef3D互換label PNG、preview PNG、体積CSVを選択式で出力できます。
- TotalSegmentator標準のROI別NIfTIマスクは、常に `totalsegmentator_original_masks/` にコピーして保持します。
- DICOMシリーズはZIP化してアップロードしてください。NIfTI `.nii/.nii.gz`、NRRD `.nrrd` も入力できます。

SegRef3D互換label PNGは、`label_png/mask0001.png` の形式で、**8-bit single-channel、背景=0、選択ROI順に Obj1=1, Obj2=2 ... Obj20=20** として出力します。


In [1]:
#@title 1. Install dependencies
!pip -q install TotalSegmentator nibabel SimpleITK numpy scikit-image trimesh svgwrite pillow gradio pandas pynrrd

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.1/293.1 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.7/100.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 97.3 MB/s eta 0:00:

In [2]:
#@title 2. Imports and ROI catalogs
from __future__ import annotations

import os, sys, re, json, shutil, zipfile, subprocess, tempfile, time
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import nibabel as nib
from skimage import measure
# import trimesh
import pandas as pd
from PIL import Image

try:
    import SimpleITK as sitk
except Exception:
    sitk = None

try:
    import nrrd
except Exception:
    nrrd = None

import gradio as gr

ROI_CATALOGS = {
  "abdominal_muscles": [
    "pectoralis_major_right",
    "pectoralis_major_left",
    "rectus_abdominis_right",
    "rectus_abdominis_left",
    "serratus_anterior_right",
    "serratus_anterior_left",
    "latissimus_dorsi_right",
    "latissimus_dorsi_left",
    "trapezius_right",
    "trapezius_left",
    "external_oblique_right",
    "external_oblique_left",
    "internal_oblique_right",
    "internal_oblique_left",
    "erector_spinae_right",
    "erector_spinae_left",
    "transversospinalis_right",
    "transversospinalis_left",
    "psoas_major_right",
    "psoas_major_left",
    "quadratus_lumborum_right",
    "quadratus_lumborum_left"
  ],
  "appendicular_bones": [
    "patella",
    "tibia",
    "fibula",
    "tarsal",
    "metatarsal",
    "phalanges_feet",
    "ulna",
    "radius",
    "carpal",
    "metacarpal",
    "phalanges_hand"
  ],
  "appendicular_bones_mr": [
    "patella",
    "tibia",
    "fibula",
    "tarsal",
    "metatarsal",
    "phalanges_feet",
    "ulna",
    "radius"
  ],
  "body": [
    "body",
    "body_trunc",
    "body_extremities",
    "skin"
  ],
  "body_mr": [
    "body_trunc",
    "body_extremities"
  ],
  "brain_structures": [
    "brainstem",
    "subarachnoid_space",
    "venous_sinuses",
    "septum_pellucidum",
    "cerebellum",
    "caudate_nucleus",
    "lentiform_nucleus",
    "insular_cortex",
    "internal_capsule",
    "ventricle",
    "central_sulcus",
    "frontal_lobe",
    "parietal_lobe",
    "occipital_lobe",
    "temporal_lobe",
    "thalamus"
  ],
  "breasts": [
    "breast"
  ],
  "cerebral_bleed": [
    "intracerebral_hemorrhage"
  ],
  "coronary_arteries": [
    "coronary_arteries"
  ],
  "craniofacial_structures": [
    "mandible",
    "teeth_lower",
    "skull",
    "head",
    "sinus_maxillary",
    "sinus_frontal",
    "teeth_upper"
  ],
  "ct": [
    "brain",
    "skull",
    "eye_left",
    "eye_right",
    "lens_left",
    "lens_right",
    "pituitary",
    "thyroid",
    "parotid_gland_left",
    "parotid_gland_right",
    "submandibular_gland_left",
    "submandibular_gland_right",
    "tongue",
    "pharynx",
    "larynx",
    "trachea",
    "esophagus",
    "lung_left",
    "lung_right",
    "heart",
    "aorta",
    "pulmonary_artery",
    "inferior_vena_cava",
    "superior_vena_cava",
    "portal_vein",
    "liver",
    "gallbladder",
    "spleen",
    "pancreas",
    "stomach",
    "duodenum",
    "small_bowel",
    "colon",
    "rectum",
    "kidney_left",
    "kidney_right",
    "adrenal_gland_left",
    "adrenal_gland_right",
    "urinary_bladder",
    "prostate",
    "seminal_vesicles",
    "uterus",
    "ovary_left",
    "ovary_right",
    "psoas_major_left",
    "psoas_major_right",
    "gluteus_maximus_left",
    "gluteus_maximus_right",
    "iliacus_left",
    "iliacus_right",
    "erector_spinae_left",
    "erector_spinae_right",
    "diaphragm",
    "hyoid",
    "clavicle_left",
    "clavicle_right",
    "scapula_left",
    "scapula_right",
    "sternum",
    "rib_left_1",
    "rib_left_2",
    "rib_left_3",
    "rib_left_4",
    "rib_left_5",
    "rib_left_6",
    "rib_left_7",
    "rib_left_8",
    "rib_left_9",
    "rib_left_10",
    "rib_left_11",
    "rib_left_12",
    "rib_right_1",
    "rib_right_2",
    "rib_right_3",
    "rib_right_4",
    "rib_right_5",
    "rib_right_6",
    "rib_right_7",
    "rib_right_8",
    "rib_right_9",
    "rib_right_10",
    "rib_right_11",
    "rib_right_12",
    "vertebrae_C1",
    "vertebrae_C2",
    "vertebrae_C3",
    "vertebrae_C4",
    "vertebrae_C5",
    "vertebrae_C6",
    "vertebrae_C7",
    "vertebrae_T1",
    "vertebrae_T2",
    "vertebrae_T3",
    "vertebrae_T4",
    "vertebrae_T5",
    "vertebrae_T6",
    "vertebrae_T7",
    "vertebrae_T8",
    "vertebrae_T9",
    "vertebrae_T10",
    "vertebrae_T11",
    "vertebrae_T12",
    "vertebrae_L1",
    "vertebrae_L2",
    "vertebrae_L3",
    "vertebrae_L4",
    "vertebrae_L5",
    "sacrum",
    "coccyx",
    "humerus_left",
    "humerus_right",
    "radius_left",
    "radius_right",
    "ulna_left",
    "ulna_right",
    "hand_left",
    "hand_right",
    "femur_left",
    "femur_right",
    "patella_left",
    "patella_right",
    "tibia_left",
    "tibia_right",
    "fibula_left",
    "fibula_right",
    "foot_left",
    "foot_right",
    "hip_left",
    "hip_right",
    "ilium_left",
    "ilium_right",
    "ischium_left",
    "ischium_right",
    "pubis_left",
    "pubis_right",
    "acetabulum_left",
    "acetabulum_right",
    "common_carotid_left",
    "common_carotid_right",
    "internal_carotid_left",
    "internal_carotid_right",
    "external_carotid_left",
    "external_carotid_right",
    "subclavian_artery_left",
    "subclavian_artery_right",
    "common_iliac_artery_left",
    "common_iliac_artery_right",
    "external_iliac_artery_left",
    "external_iliac_artery_right",
    "common_iliac_vein_left",
    "common_iliac_vein_right",
    "renal_artery_left",
    "renal_artery_right",
    "renal_vein_left",
    "renal_vein_right"
  ],
  "face": [
    "face_region"
  ],
  "face_mr": [
    "face_region"
  ],
  "head": [
    "mandible",
    "teeth_lower",
    "skull",
    "head",
    "sinus_maxillary",
    "sinus_frontal",
    "teeth_upper"
  ],
  "head_glands_cavities": [
    "eye_left",
    "eye_right",
    "eye_lens_left",
    "eye_lens_right",
    "optic_nerve_left",
    "optic_nerve_right",
    "parotid_gland_left",
    "parotid_gland_right",
    "submandibular_gland_right",
    "submandibular_gland_left",
    "nasopharynx",
    "oropharynx",
    "hypopharynx",
    "nasal_cavity_right",
    "nasal_cavity_left",
    "auditory_canal_right",
    "auditory_canal_left",
    "soft_palate",
    "hard_palate"
  ],
  "head_muscles": [
    "masseter_right",
    "masseter_left",
    "temporalis_right",
    "temporalis_left",
    "lateral_pterygoid_right",
    "lateral_pterygoid_left",
    "medial_pterygoid_right",
    "medial_pterygoid_left",
    "tongue",
    "digastric_right",
    "digastric_left"
  ],
  "headneck_bones_vessels": [
    "larynx_air",
    "thyroid_cartilage",
    "hyoid",
    "cricoid_cartilage",
    "zygomatic_arch_right",
    "zygomatic_arch_left",
    "styloid_process_right",
    "styloid_process_left",
    "internal_carotid_artery_right",
    "internal_carotid_artery_left",
    "internal_jugular_vein_right",
    "internal_jugular_vein_left"
  ],
  "headneck_muscles": [
    "sternocleidomastoid_right",
    "sternocleidomastoid_left",
    "superior_pharyngeal_constrictor",
    "middle_pharyngeal_constrictor",
    "inferior_pharyngeal_constrictor",
    "trapezius_right",
    "trapezius_left",
    "platysma_right",
    "platysma_left",
    "levator_scapulae_right",
    "levator_scapulae_left",
    "anterior_scalene_right",
    "anterior_scalene_left",
    "middle_scalene_right",
    "middle_scalene_left",
    "posterior_scalene_right",
    "posterior_scalene_left",
    "sterno_thyroid_right",
    "sterno_thyroid_left",
    "thyrohyoid_right",
    "thyrohyoid_left",
    "prevertebral_right",
    "prevertebral_left"
  ],
  "heartchambers_highres": [
    "myocardium",
    "atrium_left",
    "ventricle_left",
    "atrium_right",
    "ventricle_right",
    "aorta",
    "pulmonary_artery"
  ],
  "hip_implant": [
    "hip_implant"
  ],
  "kidney_cysts": [
    "kidney_cyst_left",
    "kidney_cyst_right"
  ],
  "liver_segments": [
    "liver_segment_1",
    "liver_segment_2",
    "liver_segment_3",
    "liver_segment_4",
    "liver_segment_5",
    "liver_segment_6",
    "liver_segment_7",
    "liver_segment_8"
  ],
  "liver_segments_mr": [
    "liver_segment_1",
    "liver_segment_2",
    "liver_segment_3",
    "liver_segment_4",
    "liver_segment_5",
    "liver_segment_6",
    "liver_segment_7",
    "liver_segment_8"
  ],
  "liver_vessels": [
    "liver_vessels",
    "liver_tumor"
  ],
  "lung_nodules": [
    "lung",
    "lung_nodules"
  ],
  "lung_vessels": [
    "lung_vessels",
    "lung_trachea_bronchia"
  ],
  "mr": [
    "brain",
    "pituitary",
    "eye_left",
    "eye_right",
    "lens_left",
    "lens_right",
    "thyroid",
    "parotid_gland_left",
    "parotid_gland_right",
    "submandibular_gland_left",
    "submandibular_gland_right",
    "pharynx",
    "larynx",
    "trachea",
    "esophagus",
    "lung_left",
    "lung_right",
    "heart",
    "aorta",
    "liver",
    "gallbladder",
    "spleen",
    "pancreas",
    "stomach",
    "duodenum",
    "small_bowel",
    "colon",
    "rectum",
    "kidney_left",
    "kidney_right",
    "adrenal_gland_left",
    "adrenal_gland_right",
    "urinary_bladder",
    "prostate",
    "seminal_vesicles",
    "uterus",
    "ovary_left",
    "ovary_right",
    "psoas_major_left",
    "psoas_major_right",
    "gluteus_maximus_left",
    "gluteus_maximus_right",
    "iliacus_left",
    "iliacus_right",
    "levator_ani_left",
    "levator_ani_right",
    "obturator_internus_left",
    "obturator_internus_right",
    "diaphragm",
    "mandible",
    "clavicle_left",
    "clavicle_right",
    "scapula_left",
    "scapula_right",
    "vertebrae_C1",
    "vertebrae_C2",
    "vertebrae_C3",
    "vertebrae_C4",
    "vertebrae_C5",
    "vertebrae_C6",
    "vertebrae_C7",
    "vertebrae_T1",
    "vertebrae_T2",
    "vertebrae_T3",
    "vertebrae_T4",
    "vertebrae_T5",
    "vertebrae_T6",
    "vertebrae_T7",
    "vertebrae_T8",
    "vertebrae_T9",
    "vertebrae_T10",
    "vertebrae_T11",
    "vertebrae_T12",
    "vertebrae_L1",
    "vertebrae_L2",
    "vertebrae_L3",
    "vertebrae_L4",
    "vertebrae_L5",
    "sacrum",
    "coccyx",
    "hip_left",
    "hip_right",
    "femur_left",
    "femur_right"
  ],
  "oculomotor_muscles": [
    "skull",
    "eyeball_right",
    "lateral_rectus_muscle_right",
    "superior_oblique_muscle_right",
    "levator_palpebrae_superioris_right",
    "superior_rectus_muscle_right",
    "medial_rectus_muscle_left",
    "inferior_oblique_muscle_right",
    "inferior_rectus_muscle_right",
    "optic_nerve_left",
    "eyeball_left",
    "lateral_rectus_muscle_left",
    "superior_oblique_muscle_left",
    "levator_palpebrae_superioris_left",
    "superior_rectus_muscle_left",
    "medial_rectus_muscle_right",
    "inferior_oblique_muscle_left",
    "inferior_rectus_muscle_left",
    "optic_nerve_right"
  ],
  "pleural_pericard_effusion": [
    "pleural_effusion",
    "pericardial_effusion"
  ],
  "teeth": [
    "lower_jawbone",
    "upper_jawbone",
    "left_inferior_alveolar_canal",
    "right_inferior_alveolar_canal",
    "left_maxillary_sinus",
    "right_maxillary_sinus",
    "pharynx",
    "bridge",
    "crown",
    "implant",
    "upper_right_central_incisor_fdi11",
    "upper_right_lateral_incisor_fdi12",
    "upper_right_canine_fdi13",
    "upper_right_first_premolar_fdi14",
    "upper_right_second_premolar_fdi15",
    "upper_right_first_molar_fdi16",
    "upper_right_second_molar_fdi17",
    "upper_right_third_molar_fdi18",
    "upper_left_central_incisor_fdi21",
    "upper_left_lateral_incisor_fdi22",
    "upper_left_canine_fdi23",
    "upper_left_first_premolar_fdi24",
    "upper_left_second_premolar_fdi25",
    "upper_left_first_molar_fdi26",
    "upper_left_second_molar_fdi27",
    "upper_left_third_molar_fdi28",
    "lower_left_central_incisor_fdi31",
    "lower_left_lateral_incisor_fdi32",
    "lower_left_canine_fdi33",
    "lower_left_first_premolar_fdi34",
    "lower_left_second_premolar_fdi35",
    "lower_left_first_molar_fdi36",
    "lower_left_second_molar_fdi37",
    "lower_left_third_molar_fdi38",
    "lower_right_central_incisor_fdi41",
    "lower_right_lateral_incisor_fdi42",
    "lower_right_canine_fdi43",
    "lower_right_first_premolar_fdi44",
    "lower_right_second_premolar_fdi45",
    "lower_right_first_molar_fdi46",
    "lower_right_second_molar_fdi47",
    "lower_right_third_molar_fdi48",
    "left_mandibular_incisive_canal_fdi103",
    "right_mandibular_incisive_canal_fdi104",
    "lingual_canal",
    "upper_right_central_incisor_pulp_fdi111",
    "upper_right_lateral_incisor_pulp_fdi112",
    "upper_right_canine_pulp_fdi113",
    "upper_right_first_premolar_pulp_fdi114",
    "upper_right_second_premolar_pulp_fdi115",
    "upper_right_first_molar_pulp_fdi116",
    "upper_right_second_molar_pulp_fdi117",
    "upper_right_third_molar_pulp_fdi118",
    "upper_left_central_incisor_pulp_fdi121",
    "upper_left_lateral_incisor_pulp_fdi122",
    "upper_left_canine_pulp_fdi123",
    "upper_left_first_premolar_pulp_fdi124",
    "upper_left_second_premolar_pulp_fdi125",
    "upper_left_first_molar_pulp_fdi126",
    "upper_left_second_molar_pulp_fdi127",
    "upper_left_third_molar_pulp_fdi128",
    "lower_left_central_incisor_pulp_fdi131",
    "lower_left_lateral_incisor_pulp_fdi132",
    "lower_left_canine_pulp_fdi133",
    "lower_left_first_premolar_pulp_fdi134",
    "lower_left_second_premolar_pulp_fdi135",
    "lower_left_first_molar_pulp_fdi136",
    "lower_left_second_molar_pulp_fdi137",
    "lower_left_third_molar_pulp_fdi138",
    "lower_right_central_incisor_pulp_fdi141",
    "lower_right_lateral_incisor_pulp_fdi142",
    "lower_right_canine_pulp_fdi143",
    "lower_right_first_premolar_pulp_fdi144",
    "lower_right_second_premolar_pulp_fdi145",
    "lower_right_first_molar_pulp_fdi146",
    "lower_right_second_molar_pulp_fdi147",
    "lower_right_third_molar_pulp_fdi148"
  ],
  "thigh_shoulder_muscles": [
    "quadriceps_femoris_left",
    "quadriceps_femoris_right",
    "thigh_medial_compartment_left",
    "thigh_medial_compartment_right",
    "thigh_posterior_compartment_left",
    "thigh_posterior_compartment_right",
    "sartorius_left",
    "sartorius_right",
    "deltoid",
    "supraspinatus",
    "infraspinatus",
    "subscapularis",
    "coracobrachial",
    "trapezius",
    "pectoralis_minor",
    "serratus_anterior",
    "teres_major",
    "triceps_brachii"
  ],
  "thigh_shoulder_muscles_mr": [
    "quadriceps_femoris_left",
    "quadriceps_femoris_right",
    "thigh_medial_compartment_left",
    "thigh_medial_compartment_right",
    "thigh_posterior_compartment_left",
    "thigh_posterior_compartment_right",
    "sartorius_left",
    "sartorius_right",
    "deltoid",
    "supraspinatus",
    "infraspinatus",
    "subscapularis",
    "coracobrachial",
    "trapezius",
    "pectoralis_minor",
    "serratus_anterior",
    "teres_major",
    "triceps_brachii"
  ],
  "tissue_4_types": [
    "subcutaneous_fat",
    "torso_fat",
    "skeletal_muscle",
    "intermuscular_fat"
  ],
  "tissue_types": [
    "subcutaneous_fat",
    "torso_fat",
    "skeletal_muscle"
  ],
  "tissue_types_mr": [
    "subcutaneous_fat",
    "torso_fat",
    "skeletal_muscle"
  ],
  "vertebrae_body": [
    "vertebrae_body",
    "intervertebral_discs"
  ],
  "vertebrae_mr": [
    "sacrum",
    "vertebrae_L5",
    "vertebrae_L4",
    "vertebrae_L3",
    "vertebrae_L2",
    "vertebrae_L1",
    "vertebrae_T12",
    "vertebrae_T11",
    "vertebrae_T10",
    "vertebrae_T9",
    "vertebrae_T8",
    "vertebrae_T7",
    "vertebrae_T6",
    "vertebrae_T5",
    "vertebrae_T4",
    "vertebrae_T3",
    "vertebrae_T2",
    "vertebrae_T1",
    "vertebrae_C7",
    "vertebrae_C6",
    "vertebrae_C5",
    "vertebrae_C4",
    "vertebrae_C3",
    "vertebrae_C2",
    "vertebrae_C1"
  ]
}

OPEN_TASKS = [
    "total", "total_mr",
    "craniofacial_structures",
    "head_glands_cavities", "head_muscles", "headneck_bones_vessels",
    "headneck_muscles", "oculomotor_muscles",
    "body", "body_mr",
    "lung_vessels", "lung_nodules", "pleural_pericard_effusion",
    "liver_vessels", "liver_segments", "liver_segments_mr",
    "kidney_cysts", "breasts", "abdominal_muscles",
    "vertebrae_mr", "hip_implant", "teeth",
]

LICENSED_TASKS = [
    "heartchambers_highres", "appendicular_bones", "appendicular_bones_mr",
    "tissue_types", "tissue_types_mr", "tissue_4_types",
    "brain_structures", "vertebrae_body",
    "face", "face_mr",
    "thigh_shoulder_muscles", "thigh_shoulder_muscles_mr",
    "coronary_arteries",
]

ALL_TASKS = ["Auto (CT→total / MRI→total_mr)"] + OPEN_TASKS + ["🔒 " + t for t in LICENSED_TASKS]

def clean_task_name(task_label: str, modality: str) -> str:
    if not task_label or task_label.startswith("Auto"):
        return "total_mr" if modality == "MRI" else "total"
    return task_label.replace("🔒 ", "").strip()

def get_roi_catalog(task_label: str, modality: str) -> List[str]:
    task = clean_task_name(task_label, modality)
    if task in ROI_CATALOGS:
        return sorted(set(ROI_CATALOGS[task]))
    if task == "total":
        return sorted(set(ROI_CATALOGS.get("ct", [])))
    if task == "total_mr":
        return sorted(set(ROI_CATALOGS.get("mr", [])))
    # fallback
    return sorted(set(ROI_CATALOGS.get("ct" if modality == "CT" else "mr", [])))

def build_roi_task_index() -> pd.DataFrame:
    rows = []
    for task, rois in ROI_CATALOGS.items():
        for roi in rois:
            rows.append({"roi": roi, "catalog/task": task})
    return pd.DataFrame(rows).sort_values(["roi", "catalog/task"]).reset_index(drop=True)

ROI_TASK_TABLE = build_roi_task_index()
print(f"Loaded ROI catalogs: {len(ROI_CATALOGS)} files / {ROI_TASK_TABLE['roi'].nunique()} unique ROI names")


Loaded ROI catalogs: 37 files / 413 unique ROI names


In [3]:
#@title 3. Helper functions

def which_totalseg() -> str:
    exe = shutil.which("TotalSegmentator") or shutil.which("TotalSegmentator.exe")
    if not exe:
        raise FileNotFoundError("TotalSegmentator CLI not found. Try re-running the install cell.")
    return exe


def prepare_input(uploaded_files, work_dir: Path) -> Path:
    """Gradio uploaded files -> NIfTI file or DICOM folder path."""
    if not uploaded_files:
        raise ValueError("Please upload a DICOM ZIP, NIfTI, or NRRD file.")

    # Gradio returns list[NamedString] or list[str] depending on version
    paths = [Path(getattr(f, "name", f)) for f in uploaded_files]

    # If one ZIP is uploaded, extract it and return extracted folder
    if len(paths) == 1 and paths[0].suffix.lower() == ".zip":
        dicom_dir = work_dir / "dicom_unzipped"
        dicom_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(paths[0], "r") as zf:
            zf.extractall(dicom_dir)
        return dicom_dir

    # If one medical image file is uploaded
    if len(paths) == 1:
        p = paths[0]
        name = p.name.lower()
        if name.endswith(".nii") or name.endswith(".nii.gz"):
            dst = work_dir / p.name
            shutil.copyfile(p, dst)
            return dst
        if name.endswith(".nrrd"):
            dst = work_dir / p.name
            shutil.copyfile(p, dst)
            return convert_nrrd_to_nifti(dst, work_dir)

    # Multiple uploaded files are treated as one DICOM series folder
    dicom_dir = work_dir / "dicom_files"
    dicom_dir.mkdir(parents=True, exist_ok=True)
    for p in paths:
        shutil.copyfile(p, dicom_dir / p.name)
    return dicom_dir


def convert_nrrd_to_nifti(nrrd_path: Path, work_dir: Path) -> Path:
    out_nii = work_dir / "input_from_nrrd.nii.gz"
    if sitk is not None:
        img = sitk.ReadImage(str(nrrd_path))
        sitk.WriteImage(img, str(out_nii))
        return out_nii
    if nrrd is None:
        raise RuntimeError("NRRD conversion requires SimpleITK or pynrrd.")
    data, hdr = nrrd.read(str(nrrd_path))
    aff = np.eye(4)
    dirs = hdr.get("space directions")
    origin = hdr.get("space origin", [0,0,0])
    try:
        aff[:3, :3] = np.array([[d[0], d[1], d[2]] for d in dirs], dtype=float)
        aff[:3, 3] = np.array(origin, dtype=float)
    except Exception:
        pass
    nib.save(nib.Nifti1Image(np.asarray(data), aff), str(out_nii))
    return out_nii


def run_totalsegmentator(
    inp: Path,
    out_dir: Path,
    task: str,
    rois: List[str],
    device: str,
    fast: bool,
    robust_crop: bool
) -> Dict[str, Path]:

    exe = which_totalseg()
    seg_dir = out_dir / "segmentations"

    args = [
        exe,
        "-i", str(inp),
        "-o", str(seg_dir),
        "-d", device,
        "--task", task,
        "--nr_thr_resamp", "1",
        "--nr_thr_saving", "1"
    ]

    # roi_subset is supported only by total / total_mr.
    # Subtasks such as craniofacial_structures must be run without roi_subset.
    allow_subset = task in ("total", "total_mr")

    if fast:
        args.append("--fast")

    if rois and allow_subset:
        args += ["--roi_subset"] + rois
        if robust_crop:
            args.append("--robust_crop")

    print("[CMD]", " ".join(map(str, args)))

    res = subprocess.run(args, capture_output=True, text=True)

    print("[I] TotalSegmentator return code:", res.returncode)

    if res.stdout:
        print(res.stdout)

    if res.stderr:
        print(res.stderr)

    print("[I] Output files after TotalSegmentator:")
    if seg_dir.exists():
        for p in sorted(seg_dir.rglob("*"))[:200]:
            print("   ", p)
    else:
        print("   seg_dir does not exist:", seg_dir)

    # If --roi_subset is rejected for this task, retry once without it.
    if res.returncode != 0 and "--roi_subset" in args:
        stderr_text = (res.stderr or "") + "\n" + (res.stdout or "")
        if "roi_subset" in stderr_text.lower() or "unrecognized" in stderr_text.lower():
            print("[W] --roi_subset seems unsupported for this task. Retrying without --roi_subset.")

            args_retry = []
            skip = False
            for a in args:
                if skip:
                    # skip ROI names after --roi_subset until next option
                    if str(a).startswith("-"):
                        skip = False
                        args_retry.append(a)
                    continue

                if a == "--roi_subset":
                    skip = True
                    continue

                args_retry.append(a)

            print("[CMD retry]", " ".join(map(str, args_retry)))
            res = subprocess.run(args_retry, capture_output=True, text=True)

            if res.stdout:
                print(res.stdout)

            if res.stderr:
                print(res.stderr)

    # Search output NIfTI files even when TotalSegmentator returns non-zero.
    # Some runs may generate valid masks but still return a warning-like non-zero code.
    nii_files = (
        list(seg_dir.rglob("*.nii")) +
        list(seg_dir.rglob("*.nii.gz"))
    )

    if res.returncode != 0:
        if nii_files:
            print(
                "[W] TotalSegmentator returned a non-zero exit code, "
                "but NIfTI mask files were found. Continuing with available masks."
            )
        else:
            raise RuntimeError(
                "TotalSegmentator failed and no segmentation NIfTI files were found. "
                "See log above."
            )

    if not nii_files:
        raise RuntimeError("No segmentation NIfTI files were found.")

    def roi_key_from_path(p: Path) -> str:
        name = p.name
        if name.endswith(".nii.gz"):
            return name[:-7]
        if name.endswith(".nii"):
            return name[:-4]
        return p.stem

    found = {roi_key_from_path(p): p for p in nii_files}
    if not rois:
        return found

    # For non-subset tasks, post-filter by selected ROI names.
    out = {}
    for r in rois:
        if r in found:
            out[r] = found[r]
            continue
        # loose fallback for nested / prefixed names
        cands = [p for p in nii_files if r.lower() in p.name.lower()]
        if cands:
            out[r] = cands[0]
    if not out:
        raise FileNotFoundError(f"Selected ROI masks were not found. Selected={rois}; available examples={list(found)[:30]}")
    return out


def write_binary_stl(stl_path: Path, vertices: np.ndarray, faces: np.ndarray):
    """
    Minimal binary STL writer.
    This avoids trimesh/scipy dependency issues in Google Colab.
    """
    stl_path = Path(stl_path)
    stl_path.parent.mkdir(parents=True, exist_ok=True)

    vertices = np.asarray(vertices, dtype=np.float32)
    faces = np.asarray(faces, dtype=np.int32)

    with open(stl_path, "wb") as f:
        header = b"Created by Instant3D Colab without trimesh"
        header = header[:80].ljust(80, b" ")
        f.write(header)

        f.write(np.uint32(len(faces)).tobytes())

        for tri in faces:
            v1, v2, v3 = vertices[tri[0]], vertices[tri[1]], vertices[tri[2]]

            normal = np.cross(v2 - v1, v3 - v1)
            norm = np.linalg.norm(normal)
            if norm > 0:
                normal = normal / norm
            else:
                normal = np.array([0.0, 0.0, 0.0], dtype=np.float32)

            f.write(np.asarray(normal, dtype=np.float32).tobytes())
            f.write(np.asarray(v1, dtype=np.float32).tobytes())
            f.write(np.asarray(v2, dtype=np.float32).tobytes())
            f.write(np.asarray(v3, dtype=np.float32).tobytes())
            f.write(np.uint16(0).tobytes())


def mask_to_stl(nifti_img, stl_path: Path, smooth_iters: int = 0):
    """
    Convert a binary NIfTI mask to STL.

    Note:
    - This Colab-safe version does not use trimesh.
    - smooth_iters is kept as an argument for GUI compatibility,
      but smoothing is currently skipped.
    """
    print(f"[I] Creating STL: {stl_path}")

    data = np.asarray(nifti_img.get_fdata())
    mask = (data > 0.5).astype(np.uint8)

    if np.count_nonzero(mask) == 0:
        raise ValueError("Cannot create STL because the mask is empty.")

    sx, sy, sz = nifti_img.header.get_zooms()[:3]

    verts, faces, normals, values = measure.marching_cubes(
        mask,
        level=0.5,
        spacing=(sx, sy, sz)
    )

    write_binary_stl(stl_path, verts, faces)

    print(f"[OK] STL saved: {stl_path}")


def save_binary_nifti(mask_img: nib.Nifti1Image, out_path: Path, foreground_value: int = 255):
    img = nib.as_closest_canonical(mask_img)
    arr = (np.asarray(img.get_fdata()) > 0.5).astype(np.uint8) * np.uint8(foreground_value)
    out = nib.Nifti1Image(arr, img.affine, img.header)
    out.header.set_data_dtype(np.uint8)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    nib.save(out, str(out_path))


def save_volume_csv(rows: List[dict], csv_path: Path):
    df = pd.DataFrame(rows)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")


def calc_volume_row(roi: str, img: nib.Nifti1Image) -> dict:
    arr = np.asarray(img.get_fdata()) > 0.5
    voxels = int(np.count_nonzero(arr))
    sx, sy, sz = img.header.get_zooms()[:3]
    voxel_mm3 = float(sx * sy * sz)
    return {
        "roi": roi,
        "voxels_count": voxels,
        "spacing_x_mm": sx,
        "spacing_y_mm": sy,
        "spacing_z_mm": sz,
        "voxel_volume_mm3": voxel_mm3,
        "total_volume_mm3": voxels * voxel_mm3,
        "total_volume_mL": voxels * voxel_mm3 / 1000.0,
    }


def oriented_volume(arr: np.ndarray, plane: str) -> np.ndarray:
    """Return volume as stack [slice, H, W] for PNG export."""
    if plane == "axial":
        return np.moveaxis(arr, 2, 0)      # z, x, y
    if plane == "coronal":
        return np.moveaxis(arr, 1, 0)      # y, x, z
    if plane == "sagittal":
        return np.moveaxis(arr, 0, 0)      # x, y, z
    return np.moveaxis(arr, 2, 0)


def save_segref_label_pngs(
    roi_imgs: Dict[str, nib.Nifti1Image],
    out_dir: Path,
    plane="axial",
    reverse=False,
    flip_lr=False,
    flip_ud=True,
    rotate90=False
):
    """Save SegRef3D-compatible canonical label PNGs and preview PNGs.
    0=background, selected ROI order -> Obj1=1, Obj2=2 ... Obj20=20.
    """
    if len(roi_imgs) > 20:
        print("[W] SegRef3D label PNG supports Obj1..Obj20. Only first 20 ROIs are written.")

    rois = list(roi_imgs.keys())[:20]
    stacks = []

    for roi in rois:
        img = nib.as_closest_canonical(roi_imgs[roi])
        arr = (np.asarray(img.get_fdata()) > 0.5).astype(np.uint8)
        stacks.append(oriented_volume(arr, plane))

    n = stacks[0].shape[0]
    label_dir = out_dir / "label_png"
    preview_dir = out_dir / "preview_png"
    label_dir.mkdir(parents=True, exist_ok=True)
    preview_dir.mkdir(parents=True, exist_ok=True)

    colors_rgb = [
        (255,0,0),(0,0,255),(0,255,0),(255,255,0),(128,0,128),
        (255,165,0),(0,255,255),(173,255,47),(128,128,128),(0,128,128),
        (255,192,203),(255,20,147),(0,128,0),(128,0,0),(0,255,230),
        (255,215,0),(255,69,0),(0,0,128),(220,20,60),(128,128,0),
    ]

    order = range(n-1, -1, -1) if reverse else range(n)

    for out_i, src_i in enumerate(order, start=1):
        label = np.zeros(stacks[0][src_i].shape, dtype=np.uint8)

        for obj_id, stack in enumerate(stacks, start=1):
            mask2d = stack[src_i]
            label[mask2d > 0] = obj_id

        # まず回転する
        if rotate90:
            label = np.rot90(label, k=-1)   # clockwise
            # 逆だったら k=1 に変更

        # 回転後の最終画像に対して左右・上下反転をかける
        if flip_lr:
            label = np.flip(label, axis=1)

        if flip_ud:
            label = np.flip(label, axis=0)

        print(
            f"[DEBUG SegRef PNG] out_i={out_i:04d}, src_i={src_i}, "
            f"plane={plane}, reverse={reverse}, "
            f"rotate90={rotate90}, flip_lr={flip_lr}, flip_ud={flip_ud}, "
            f"label_shape={label.shape}, unique={np.unique(label).tolist()}"
        )

        Image.fromarray(label).save(label_dir / f"mask{out_i:04d}.png")

        preview = np.zeros((*label.shape, 3), dtype=np.uint8)
        for obj_id, (r, g, b) in enumerate(colors_rgb, start=1):
            preview[label == obj_id] = (r, g, b)

        Image.fromarray(preview).save(preview_dir / f"preview_mask{out_i:04d}.png")

    pd.DataFrame(
        [{"object_id": i+1, "roi": r} for i, r in enumerate(rois)]
    ).to_csv(out_dir / "segref_object_mapping.csv", index=False, encoding="utf-8-sig")

def zip_dir(folder: Path, zip_path: Path) -> Path:
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_path.with_suffix("")), "zip", str(folder))
    return zip_path


In [ ]:
#@title 4. Gradio GUI

def update_roi_choices(modality, task_label):
    rois = get_roi_catalog(task_label, modality)
    table = ROI_TASK_TABLE.copy()
    if task_label and not task_label.startswith("Auto"):
        task = clean_task_name(task_label, modality)
        table = table[table["catalog/task"].eq(task)]
    else:
        fallback = "mr" if modality == "MRI" else "ct"
        table = table[table["catalog/task"].eq(fallback)]
    return gr.update(choices=rois, value=[]), table


def instant3d_run(
    uploaded_files,
    modality,
    task_label,
    selected_rois,
    extra_rois_text,
    device_choice,
    fast,
    robust_crop,
    export_stl,
    export_segref_png,
    export_volume_csv,
    plane,
    reverse_slices,
    flip_lr,
    flip_ud,
    smooth_iters,
):
    log_lines = []
    def log(msg):
        print(msg)
        log_lines.append(str(msg))

    try:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        work_dir = Path("/content") / f"instant3d_work_{timestamp}"
        out_dir = Path("/content") / f"Instant3D_output_{timestamp}"
        work_dir.mkdir(parents=True, exist_ok=True)
        out_dir.mkdir(parents=True, exist_ok=True)

        inp = prepare_input(uploaded_files, work_dir)
        task = clean_task_name(task_label, modality)

        rois = list(selected_rois or [])
        if extra_rois_text and extra_rois_text.strip():
            rois += [r.strip() for r in re.split(r"[,\n\s]+", extra_rois_text) if r.strip()]
        # preserve order, remove duplicates
        seen = set()
        rois = [r for r in rois if not (r in seen or seen.add(r))]

        if not rois:
            raise ValueError("Please select at least one ROI. Running all ROIs is intentionally disabled for Colab to avoid huge outputs.")

        valid = set(get_roi_catalog(task_label, modality))
        bad = [r for r in rois if r not in valid]
        if bad:
            raise ValueError(f"Invalid ROI name(s) for this task: {bad}. Use the ROI dropdown/search list.")

        device = "cpu" if device_choice == "CPU" else "gpu"
        log(f"Input: {inp}")
        log(f"Task: {task}")
        log(f"Selected ROIs: {rois}")
        log(f"Device: {device}")

        roi_paths = run_totalsegmentator(inp, out_dir, task, rois, device, fast, robust_crop)
        roi_imgs = {roi: nib.as_closest_canonical(nib.load(str(path))) for roi, path in roi_paths.items()}

        # Keep original TotalSegmentator masks as reference
        original_dir = out_dir / "totalsegmentator_original_masks"
        original_dir.mkdir(exist_ok=True)

        for roi, p in roi_paths.items():
            shutil.copyfile(p, original_dir / p.name)

        log(f"Original TotalSegmentator masks saved to: {original_dir}")

        if not export_stl and not export_segref_png and not export_volume_csv:
            log("No optional outputs selected. Only original TotalSegmentator NIfTI masks will be included in the ZIP.")

        volume_rows = []

        for roi, img in roi_imgs.items():
            log(f"Post-processing: {roi}")
            volume_rows.append(calc_volume_row(roi, img))

            if export_stl:
                stl_path = out_dir / "stl" / f"{roi}_mesh.stl"
                mask_to_stl(img, stl_path, int(smooth_iters))

        volume_rows = []
        for roi, img in roi_imgs.items():
            log(f"Post-processing: {roi}")
            volume_rows.append(calc_volume_row(roi, img))

            if export_stl:
                stl_path = out_dir / "stl" / f"{roi}_mesh.stl"
                mask_to_stl(img, stl_path, int(smooth_iters))

        if export_segref_png:
            save_segref_label_pngs(
                roi_imgs,
                out_dir / "segref3d_compatible_png",
                plane=plane,
                reverse=reverse_slices,
                flip_lr=flip_lr,
                flip_ud=True,      # ← まず強制ONで確認
                rotate90=True
            )

        if export_volume_csv:
            save_volume_csv(volume_rows, out_dir / "volume_summary.csv")

        # Always write run summary
        pd.DataFrame({"selected_roi": rois}).to_csv(out_dir / "selected_rois.csv", index=False, encoding="utf-8-sig")
        with open(out_dir / "run_log.txt", "w", encoding="utf-8") as f:
            f.write("\n".join(log_lines))

        zip_path = zip_dir(out_dir, Path("/content") / f"Instant3D_output_{timestamp}.zip")
        log(f"Done: {zip_path}")
        return str(zip_path), "\n".join(log_lines)

    except Exception as e:
        log(f"ERROR: {e}")
        return None, "\n".join(log_lines)


with gr.Blocks(title="Instant3Dweb - TotalSegmentator GUI") as demo:
    gr.Markdown("""
    # Instant3Dweb - TotalSegmentator GUI

    **Instructions:**

    1. DICOM ZIP / NIfTI / NRRD をアップロード
    2. taskを選ぶ
    3. ROI欄で候補を検索して複数選択
    4. 出力形式を選んで Run

    ※ ROI名はTotalSegmentatorの正式名称です。`kidney`ではなく `kidney_left`, `kidney_right` のように選択してください。

    ※ TotalSegmentator標準の対象物別NIfTIマスクは `totalsegmentator_original_masks/` に保存されます。
    """)

    with gr.Row():
        uploaded = gr.File(label="Input files: DICOM ZIP, NIfTI, NRRD, or multiple DICOM files", file_count="multiple")

    with gr.Row():
        modality = gr.Radio(["CT", "MRI"], value="CT", label="Modality")
        task = gr.Dropdown(choices=ALL_TASKS, value="Auto (CT→total / MRI→total_mr)", label="TotalSegmentator task")
        device = gr.Radio(["GPU", "CPU"], value="GPU", label="Device")

    roi_dropdown = gr.Dropdown(
        choices=get_roi_catalog("Auto (CT→total / MRI→total_mr)", "CT"),
        value=[],
        multiselect=True,
        label="Target ROI(s) — type to search / multiple selection",
        info="候補欄に途中まで入力すると絞り込みできます。",
    )
    extra_rois = gr.Textbox(label="Optional manual ROI input", placeholder="例: liver kidney_right（通常は上の候補選択を推奨）", lines=2)
    roi_table = gr.Dataframe(value=ROI_TASK_TABLE[ROI_TASK_TABLE["catalog/task"].eq("ct")], label="ROI catalog for selected task/modality", interactive=False)

    modality.change(update_roi_choices, inputs=[modality, task], outputs=[roi_dropdown, roi_table])
    task.change(update_roi_choices, inputs=[modality, task], outputs=[roi_dropdown, roi_table])

    with gr.Accordion("Execution options", open=False):
        fast = gr.Checkbox(value=False, label="Fast mode (--fast, 3mm model)")
        robust_crop = gr.Checkbox(value=False, label="Robust crop for roi_subset")

    with gr.Accordion("Output options", open=True):

        gr.Markdown(
            """
            **Note:** TotalSegmentator original NIfTI masks are always saved.

            These checkboxes control only additional outputs:
            STL mesh, SegRef3D-compatible label PNG, and volume CSV.
            """
        )

        export_stl = gr.Checkbox(
            label="Export STL mesh",
            value=False
        )

        export_segref_png = gr.Checkbox(
            label="Export SegRef3D-compatible canonical label PNG",
            value=False
        )

        export_volume_csv = gr.Checkbox(
            label="Export volume CSV",
            value=False
        )

    with gr.Accordion("SegRef3D label PNG orientation", open=False):
        plane = gr.Radio(["axial", "coronal", "sagittal"], value="axial", label="Slice plane")
        reverse_slices = gr.Checkbox(value=False, label="Reverse slice order")
        flip_lr = gr.Checkbox(value=False, label="Flip L/R")
        flip_ud = gr.Checkbox(value=False, label="Flip U/D")

    smooth_iters = gr.Slider(0, 100, value=10, step=1, label="STL smoothing iterations")
    run_btn = gr.Button("Run Instant3D", variant="primary")
    output_zip = gr.File(label="Download output ZIP")
    logbox = gr.Textbox(label="Log", lines=16)

    run_btn.click(
        instant3d_run,
        inputs=[uploaded, modality, task, roi_dropdown, extra_rois, device, fast, robust_crop,
                export_stl, export_segref_png,
                export_volume_csv, plane, reverse_slices, flip_lr, flip_ud, smooth_iters],
        outputs=[output_zip, logbox]
    )

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://48ef7de72bfcdd7c04.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Input: /content/instant3d_work_20260608_094546/dicom_unzipped
Task: craniofacial_structures
Selected ROIs: ['mandible']
Device: gpu
[CMD] /usr/local/bin/TotalSegmentator -i /content/instant3d_work_20260608_094546/dicom_unzipped -o /content/Instant3D_output_20260608_094546/segmentations -d gpu --task craniofacial_structures --nr_thr_resamp 1 --nr_thr_saving 1
[I] TotalSegmentator return code: 0

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

TotalSegmentator sends anonymous usage statistics. If you want to disable it check the documentation.
Download finished. Extracting...
Generating rough segmentation for cropping...
Download finished. Extracting...
Converting dicom to nifti...
  found image with shape (512, 512, 441)
Resampling...
  Resampled in 9.24s
Predicting...
Download finished. Extracting...
Converting dicom to nifti...
  found image with shape (512, 512, 441)
Resampling...
  Resampled in 9.24s
Predicting...
  Predicted in 12.52s
Resampling...
